In [6]:
import sys
!{sys.executable} -m pip install google-genai pandas

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 9.9 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 10.5 MB/s  0:00:00
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------------- -------------------- 1.8/3.8 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 9.8 MB/s  0:00:00
Using 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
import time
import random
import pandas as pd
from google import genai
from google.genai import types
API_KEYS = [
]
MASTER_CSV = "master_with_song_id.csv"          
SEMANTIC_SONGS_CSV = "data/semantic_songs.csv"
ROTATION_STATE_FILE = "data/rotation_state.json"


if os.environ.get("GEMINI_API_KEYS"):
    API_KEYS = [k.strip() for k in os.environ["GEMINI_API_KEYS"].split(",") if k.strip()]
GEMINI_MODEL = "gemini-flash-latest"   
BATCH_SIZE = 20                                  
RATE_LIMIT_SEC = 4.0                            
MAX_RETRIES = 3
RANDOM_SEED = 42
 
os.makedirs("data", exist_ok=True)
random.seed(RANDOM_SEED)


In [20]:
def load_rotation_state():
    if os.path.exists(ROTATION_STATE_FILE):
        with open(ROTATION_STATE_FILE) as f:
            return json.load(f)
    return {"current_index": 0, "exhausted_today": []}
 
def save_rotation_state(state):
    with open(ROTATION_STATE_FILE, "w") as f:
        json.dump(state, f)
 
rotation_state = load_rotation_state()
current_index = rotation_state["current_index"]
exhausted_today = set(rotation_state["exhausted_today"])
 
def build_client(idx):
    return genai.Client(api_key=API_KEYS[idx])
 
client = build_client(current_index)
print(f"Starting with API key #{current_index + 1}/{len(API_KEYS)}")
if exhausted_today:
    print(f"Keys already marked exhausted this run: {sorted(exhausted_today)}")
 
def rotate_key():
    global current_index, client
    exhausted_today.add(current_index)
    for _ in range(len(API_KEYS)):
        current_index = (current_index + 1) % len(API_KEYS)
        if current_index not in exhausted_today:
            client = build_client(current_index)
            rotation_state["current_index"] = current_index
            rotation_state["exhausted_today"] = list(exhausted_today)
            save_rotation_state(rotation_state)
            print(f"  -> switched to API key #{current_index + 1}/{len(API_KEYS)}")
            return True
    return False  # every key exhausted
 
def is_quota_error(e):
    msg = str(e)
    return "RESOURCE_EXHAUSTED" in msg or "429" in msg

Starting with API key #20/20
Keys already marked exhausted this run: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


In [21]:
SYSTEM_PROMPT = """You are helping build a music semantic-search dataset.
For each song given (name, artist(s), language, year), you must:
 
1. Write a `semantic_profile`: 3-5 sentences describing the song's
   THEME, EMOTION, and IDEAL LISTENING CONTEXT -- based on what the
   song is actually about (its lyrics/theme/mood), not on tempo or
   popularity numbers. Style example:
 
   "A timeless romantic Hindi classic sung by Lata Mangeshkar. The
   song expresses deep love, longing, fear of separation, and
   nostalgia. Ideal for peaceful evenings, late-night listening, and
   emotional moments."
 
   If you genuinely do not recognize a song and cannot infer its
   theme with reasonable confidence, write a cautious, honest profile
   based only on artist/era/genre conventions, and set
   profile_confidence to "low" (otherwise "high").

"""
 
# Structured output schema so Gemini returns clean JSON directly
RESPONSE_SCHEMA = {
    "type": "ARRAY",
    "items": {
        "type": "OBJECT",
        "properties": {
            "song_id": {"type": "INTEGER"},
            "semantic_profile": {"type": "STRING"},
            "profile_confidence": {"type": "STRING", "enum": ["high", "low"]},
        },
        "required": ["song_id", "semantic_profile", "profile_confidence"],
    },
}
 
 
def build_user_prompt(songs_batch):
    lines = []
    for s in songs_batch:
        lines.append(
            f'- song_id={s["song_id"]}, name="{s["song_name"]}", '
            f'artist="{s["primary_singer"]}", language={s["language"]}, '
            f'year={s["year"]}'
        )
    return "Songs:\n" + "\n".join(lines)
 
 
def call_llm(songs_batch):

    global client
    system = SYSTEM_PROMPT
    user = build_user_prompt(songs_batch)
    config = types.GenerateContentConfig(
        system_instruction=system,
        response_mime_type="application/json",
        response_schema=RESPONSE_SCHEMA,
        temperature=0.7,
    )
 
    keys_tried_this_call = 0
    while keys_tried_this_call < len(API_KEYS):
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=user,
                    config=config,
                )
                return json.loads(resp.text.strip())
            except Exception as e:
                if is_quota_error(e):
                    print(f"  key #{current_index + 1} hit quota -- rotating")
                    break  # stop retrying this key, go rotate below
                else:
                    wait = RATE_LIMIT_SEC * attempt
                    print(f"  [key #{current_index + 1}, attempt {attempt}] "
                          f"non-quota error: {e} -- retrying in {wait:.0f}s")
                    time.sleep(wait)
        else:
            pass
 
        keys_tried_this_call += 1
        if not rotate_key():
            print("ALL API KEYS EXHAUSTED for today. Stopping cleanly -- "
                  "just re-run this script tomorrow, it will resume from "
                  "where it left off.")
            return "ALL_KEYS_EXHAUSTED"
 
    return None  

In [22]:
df_master = pd.read_csv(MASTER_CSV)
df_master["language"] = df_master["language"].astype(str).str.strip().str.lower()
df_master["year"] = pd.to_datetime(df_master["released_date"], errors="coerce").dt.year
df_master["year"] = df_master["year"].fillna("unknown")
 
done_ids = set()
if os.path.exists(SEMANTIC_SONGS_CSV):
    done_ids = set(pd.read_csv(SEMANTIC_SONGS_CSV)["song_id"].tolist())
    print(f"Resuming: {len(done_ids)} songs already done, skipping them.")
 
df_todo = df_master[~df_master["song_id"].isin(done_ids)].reset_index(drop=True)
print(f"{len(df_todo)} songs left to process.")

Resuming: 7408 songs already done, skipping them.
268 songs left to process.


In [26]:
semantic_rows = []
 
t0 = time.time()
n_batches = (len(df_todo) + BATCH_SIZE - 1) // BATCH_SIZE
 
for b in range(n_batches):
    batch_df = df_todo.iloc[b * BATCH_SIZE: (b + 1) * BATCH_SIZE]
    songs_batch = batch_df.to_dict("records")
 
    result = call_llm(songs_batch)
 
    if result == "ALL_KEYS_EXHAUSTED":
        break  # stop the whole run, resume tomorrow
    if result is None:
        print(f"[batch {b}] failed on every key for non-quota reasons, skipping this batch.")
        continue
 
    for item in result:
        sid = item["song_id"]
        src_row = batch_df[batch_df["song_id"] == sid]
        if src_row.empty:
            continue
        src = src_row.iloc[0]
 
        semantic_rows.append({
            "song_id": sid,
            "song_name": src["song_name"],
            "primary_singer": src["primary_singer"],
            "language": src["language"],
            "semantic_profile": item.get("semantic_profile", ""),
            "profile_confidence": item.get("profile_confidence", "low"),
        })

    pd.DataFrame(semantic_rows).to_csv(
        SEMANTIC_SONGS_CSV, mode="a", index=False,
        header=not os.path.exists(SEMANTIC_SONGS_CSV),
    )
    
    semantic_rows = []
    elapsed = time.time() - t0
    done_songs = min((b + 1) * BATCH_SIZE, len(df_todo))
    eta_min = elapsed / done_songs * len(df_todo) / 60 if done_songs else 0
    print(f"batch {b+1}/{n_batches} done ({done_songs}/{len(df_todo)} songs), "
          f"key #{current_index + 1}, {elapsed:.0f}s elapsed, ~{eta_min:.1f} min ETA")
 
    time.sleep(RATE_LIMIT_SEC)
 
print("\nRun finished (or paused due to quota).")
print(f"semantic_songs.csv -> {SEMANTIC_SONGS_CSV}")


batch 1/14 done (20/268 songs), key #20, 123s elapsed, ~27.6 min ETA
  key #20 hit quota -- rotating
ALL API KEYS EXHAUSTED for today. Stopping cleanly -- just re-run this script tomorrow, it will resume from where it left off.

Run finished (or paused due to quota).
semantic_songs.csv -> data/semantic_songs.csv


In [24]:
if os.path.exists(SEMANTIC_SONGS_CSV):
    df_sem = pd.read_csv(SEMANTIC_SONGS_CSV)
    print(f"\nTotal songs with semantic profiles: {len(df_sem)}")
    low_conf = (df_sem["profile_confidence"] == "low").mean()
    print(f"Low-confidence profile rate: {low_conf:.1%}")


Total songs with semantic profiles: 7696
Low-confidence profile rate: 2.7%


In [ ]:
df = pd.read_csv("data/semantic_songs.csv")
df = df.drop(columns='song_id')
df.head(10)   

        

,song_name,primary_singer,language,semantic_profile,profile_confidence
0,bheegi bheegi raaton mein,kishore kumar,hindi,A classic romantic Hindi duet featuring Kishor...,high
1,aap ki ankhon mein kuch,kishore kumar,hindi,A charming and tender romantic classic where l...,high
2,tum aa gaye ho noor aa gaya,kishore kumar,hindi,A deeply poetic song celebrating the overwhelm...,high
3,hamen tumse pyar kitna,kishore kumar,hindi,"An iconic Hindi song describing intense, borde...",high
4,tere bina zindagi se,kishore kumar,hindi,A legendary bittersweet composition exploring ...,high
5,kora kagaz tha yeh man mera,kishore kumar,hindi,"A sweet, heartwarming classic about how true l...",high
6,lag ja gale se phir,lata mangeshkar,hindi,An unmatched masterpiece expressing intense ro...,high
7,ae mere humsafar,udit narayan,hindi,A bright and melodious 90s romantic song fille...,high
8,ek ajnabee haseena se,kishore kumar,hindi,A captivating romantic Hindi song sung by Kish...,high
9,tere mere milan ki yeh raina,kishore kumar,hindi,A profound romantic ballad featuring the iconi...,high


In [6]:
import pandas as pd
import re
# ---- 1. Drop the confidence column ----
df = df.drop(columns=["profile_confidence"], errors="ignore")
def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
    text = re.sub(r"\s+", " ", text)      # collapse multiple spaces
    return text

df["_song_key"] = df["song_name"].apply(normalize)
df["_singer_key"] = df["primary_singer"].apply(normalize)

In [7]:
df["_song_key"] = df["song_name"].apply(normalize)
df["_singer_key"] = df["primary_singer"].apply(normalize)

In [9]:
# ---- 3. Remove duplicates on (song_name + primary_singer) ----
before = len(df)
df = df.drop_duplicates(subset=["_song_key", "_singer_key"], keep="first")
after = len(df)

print(f"Removed {before - after} duplicate rows ({before} -> {after})")

# ---- 4. Drop helper columns ----
df = df.drop(columns=["_song_key", "_singer_key"])

Removed 1874 duplicate rows (7776 -> 5902)


In [10]:
df.head(10) 

,song_name,primary_singer,language,semantic_profile
0,bheegi bheegi raaton mein,kishore kumar,hindi,A classic romantic Hindi duet featuring Kishor...
1,aap ki ankhon mein kuch,kishore kumar,hindi,A charming and tender romantic classic where l...
2,tum aa gaye ho noor aa gaya,kishore kumar,hindi,A deeply poetic song celebrating the overwhelm...
3,hamen tumse pyar kitna,kishore kumar,hindi,"An iconic Hindi song describing intense, borde..."
4,tere bina zindagi se,kishore kumar,hindi,A legendary bittersweet composition exploring ...
5,kora kagaz tha yeh man mera,kishore kumar,hindi,"A sweet, heartwarming classic about how true l..."
6,lag ja gale se phir,lata mangeshkar,hindi,An unmatched masterpiece expressing intense ro...
7,ae mere humsafar,udit narayan,hindi,A bright and melodious 90s romantic song fille...
8,ek ajnabee haseena se,kishore kumar,hindi,A captivating romantic Hindi song sung by Kish...
9,tere mere milan ki yeh raina,kishore kumar,hindi,A profound romantic ballad featuring the iconi...


In [12]:
# ---- 1. Replace language 'old' with 'hindi' ----
df["language"] = df["language"].replace("old", "hindi")
df["language"] = df["language"].apply(
    lambda x: "hindi" if isinstance(x, str) and x.strip().lower() == "old" else x
)

In [16]:
df.insert(0, "song_id", range(1, len(df) + 1))

In [21]:
print(df.head())

   song_id                    song_name primary_singer language  \
0        1    bheegi bheegi raaton mein  kishore kumar    hindi   
1        2      aap ki ankhon mein kuch  kishore kumar    hindi   
2        3  tum aa gaye ho noor aa gaya  kishore kumar    hindi   
3        4       hamen tumse pyar kitna  kishore kumar    hindi   
4        5         tere bina zindagi se  kishore kumar    hindi   

                                    semantic_profile  
0  A classic romantic Hindi duet featuring Kishor...  
1  A charming and tender romantic classic where l...  
2  A deeply poetic song celebrating the overwhelm...  
3  An iconic Hindi song describing intense, borde...  
4  A legendary bittersweet composition exploring ...  


In [23]:
print(df.duplicated(subset=["song_name", "primary_singer"]).sum())




0


In [25]:
df[df["semantic_profile"].isna()]
df[df["semantic_profile"].str.strip() == ""]
df["profile_length"] = df.semantic_profile.str.split().str.len()

print(df.profile_length.describe())

count    5902.000000
mean       42.253643
std         6.137570
min        27.000000
25%        37.000000
50%        42.000000
75%        47.000000
max        66.000000
Name: profile_length, dtype: float64


In [26]:
df.to_csv("data/semantic_songs.csv", index=False)